# 04 — Viz (exploration)

**billboard-hot100.** Explore how music **genres** rose and fell at the top of the
Billboard Hot 100 over the decades, and settle the **framing** before the lead
**bump chart** gets built for real in `06-viz-social`.

> ⭐ This notebook **CONSUMES** the chart-ready tables `03-prepare` built
> (`chart_genre_by_year`, `chart_genre_by_decade`, `chart_genre_rank_top`,
> `chart_genre_number_ones_by_decade`). It opens DuckDB **read-only** and does
> **not** create any `chart_*` table. If exploration shows we need a new cut, the
> fix is to amend `03-prepare` and re-run — never to build chart data here.

**On the bump template:** the shared kit has no `bump` type yet — building it is
*this project's* capability add, scheduled for `06-viz-social`. A bump is just a
specialised line of **rank vs year**, so for exploration we draw the rank race with
the existing shared **`line`** template: we plot an **inverted** rank (so rank 1
sits at the TOP) by charting `(N+1 − rank)` and relabelling the y ticks back to
`#rank`. The goal is to let the owner SEE and choose the framing, not to produce
the final bump.

The venv is Python 3.14 (matplotlib RecursionErrors), so every chart is rendered
with the shared **Pillow factory** (`render_chart`) and displays **inline**.

In [ ]:
import sys, os
from pathlib import Path

# Navigate to project root regardless of how the kernel was launched.
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / '..' / '..' / 'shared'))

import duckdb, pandas as pd
pd.set_option('display.max_rows', 120)

from src.ingest import load_config
from colors import c, CATEGORICAL_10
from chart_factory import render_chart

cfg = load_config('config.yaml')
# READ-ONLY: 04-viz only consumes 03's prepared tables.
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
print('tables:', [r[0] for r in con.execute('SHOW TABLES').fetchall()])

## Rank-bump helper (via the `line` template)

A tiny helper that turns a long `(year, genre, rank)` frame into the wide,
inverted-rank form the `line` template wants, then renders it. We pass the
line template a `y_fmt` lambda so the axis still reads as `#1 … #N` even though
the plotted value is `(N+1 − rank)`. One **genre = one line**, labelled at its
last point. This is a stand-in for the real bump template coming in `06`.

In [ ]:
def bump_from_long(df_long, rank_col='rank', x_col='year', genre_col='genre',
                   genres=None, palette=None, title='', subtitle='', source=None,
                   filename=None, x_tick_step=10):
    """Render a rank bump (rank 1 at top) from a long (x, genre, rank) frame
    using the shared `line` template. Returns nothing; displays inline."""
    d = df_long.copy()
    if genres is None:
        genres = list(d[genre_col].unique())
    d = d[d[genre_col].isin(genres)]
    N = int(d[rank_col].max())
    # Invert: rank 1 -> top. Plot (N+1 - rank).
    d['_inv'] = (N + 1) - d[rank_col]
    wide = d.pivot_table(index=x_col, columns=genre_col, values='_inv',
                         aggfunc='mean').reset_index()
    if palette is None:
        palette = CATEGORICAL_10
    # Order series by their most-recent rank so the legend/labels read sensibly.
    last = d.sort_values(x_col).groupby(genre_col)[rank_col].last().sort_values()
    ordered = [g for g in last.index if g in wide.columns]
    series = [{'col': g, 'label': g, 'color': palette[i % len(palette)]}
              for i, g in enumerate(ordered)]
    render_chart({
        'type': 'line',
        'table': wide,
        'x_col': x_col,
        'series': series,
        'y_fmt': (lambda v, _N=N: f'#{int(_N + 1 - round(v))}'),
        'x_axis_label': x_col.capitalize(),
        'y_axis_label': 'Rank among set (1 = most chart appearances)',
        'y_min': 0.5, 'y_max': N + 0.5,
        'markers': True, 'label_last': True, 'legend': False,
        'x_tick_step': x_tick_step,
        'title': title, 'subtitle': subtitle, 'source': source,
        'filename': filename,
    })

SRC = 'Billboard Hot 100 (chart history) · genre via MusicBrainz · @unwelcomedata'
print('helper ready')

## 1. THE KEY DECISION — how to handle `Other/Unknown`

`Other/Unknown` is **not a genre** — it is the coverage residual (unmatched
artists + the long tail below the ≥ 20-appearance enrichment threshold). But it is
big: the `03` sanity check showed it ranks **#1 in the 1950s, 1960s and 1990s**.
So the bump looks very different depending on whether we show it.

We render the rank race **three ways** so the owner can compare directly:

- **(a) top-6 real genres only** — clean 6-line race from `chart_genre_rank_top`.
  Honest about which genres dominate, but the ranks are *rank among real genres*,
  so the 1950s/60s look more settled than reality (the big 'unknown' mass is hidden).
- **(b) include `Other/Unknown`** — the 6 real genres + Other = 7 lines, re-ranked
  within that set. Honest that the early decades are heavily 'unknown', but it adds
  a dominant grey line that is not a genre and crowds the top of the chart.
- **(c) top-6 real, 1970-start** — same as (a) but begins in 1970, when genre
  coverage is strong (`Other` is worst pre-1970). Sidesteps the thin-coverage era
  entirely instead of hiding or foregrounding it.

### (a) Top-6 real genres only — the clean race (`chart_genre_rank_top`)

In [ ]:
top6 = con.execute('SELECT * FROM chart_genre_rank_top ORDER BY year, rank').df()
bump_from_long(
    top6,
    title='Genre rank on the Billboard Hot 100, by year',
    subtitle='Top 6 genres by chart appearances; Other/Unknown excluded. Coverage is thin pre-1970.',
    source=SRC, filename='explore_bump_top6_year',
)

### (b) Include `Other/Unknown` as its own line (7 lines, re-ranked within the set)
Take the 6 real genres **plus** `Other/Unknown` from `chart_genre_by_year`, then
re-rank *within those seven* each year (so it's a dense 1..7 race). The grey line
is `Other/Unknown` — watch it dominate the early decades and fade as coverage
improves.

In [ ]:
seven = con.execute('''
    WITH keep AS (
        SELECT * FROM chart_genre_by_year
        WHERE genre IN (SELECT DISTINCT genre FROM chart_genre_rank_top)
           OR genre = 'Other/Unknown'
    )
    SELECT year, genre, appearances, share,
           DENSE_RANK() OVER (PARTITION BY year ORDER BY appearances DESC) AS rank
    FROM keep
    ORDER BY year, rank
''').df()
# Palette: give Other/Unknown a flat grey so it reads as 'not a genre'.
genres7 = seven.sort_values('year').groupby('genre')['rank'].last().sort_values().index.tolist()
pal7 = []
ci = 0
for g in genres7:
    if g == 'Other/Unknown':
        pal7.append(c('gray'))
    else:
        pal7.append(CATEGORICAL_10[ci % len(CATEGORICAL_10)]); ci += 1
bump_from_long(
    seven, genres=genres7, palette=pal7,
    title='Genre rank on the Billboard Hot 100, by year (with Other/Unknown)',
    subtitle='Top 6 genres + Other/Unknown (grey), re-ranked within the set. Other/Unknown is a coverage residual, not a genre.',
    source=SRC, filename='explore_bump_seven_year',
)

### (c) Top-6 real, **1970-start** — avoid the thin-coverage era
Same six lines as (a), but only from 1970 on. Ranks are re-densed within the
top-6 for the 1970+ window so the race reads cleanly where coverage is strong.

In [ ]:
top6_1970 = con.execute('''
    SELECT year, genre, appearances, share,
           DENSE_RANK() OVER (PARTITION BY year ORDER BY appearances DESC) AS rank
    FROM chart_genre_rank_top
    WHERE year >= 1970
    ORDER BY year, rank
''').df()
bump_from_long(
    top6_1970,
    title='Genre rank on the Billboard Hot 100, 1970–present',
    subtitle='Top 6 genres by chart appearances, from 1970 (when genre coverage is strong). Other/Unknown excluded.',
    source=SRC, filename='explore_bump_top6_1970',
)

**Tradeoff + Kiro recommendation.** (a) is the cleanest social chart and matches
what the piece is about — the genres that actually dominate the Hot 100 — but it
quietly hides how little we know about the 1958–1969 era. (b) is the most honest
about coverage, but a fat grey 'not-a-genre' line on top for two decades is a
weak visual lead and invites 'what is Other?'. (c) is the honest-AND-clean
compromise: it just doesn't claim to rank genres in the era where we can't.

➡️ **Kiro lean: ship (c) — top-6 real genres, 1970-start — as the lead bump**, with
a one-line footnote that pre-1970 genre coverage is too thin to rank. If the owner
wants the full 1958 span for the sweep, use (a) with that same caveat as a visible
subtitle. Avoid (b) for the social lead; keep it as the honesty backup if a critic
asks 'where's the unknown mass?'.

## 2. Count/rank vs share

The bump ranks genres by **appearances** (count). A rank-bump answers *who's on
top*; it hides *by how much*. The alternative framing is **share of the Hot 100** —
a line chart with share (%) on the y-axis — which answers *how much of the chart
each genre actually held*. The two tell slightly different stories (a genre can be
#1 by a hair, or #2 by a mile), so we show both.

### 2a. Confirm the rank bump matches the `rank` column
Sanity check: the helper re-pivots from the `rank` column `03` already computed, so
the top-6 bump IS the stored rank. Spot-check a couple of years.

In [ ]:
chk = con.execute('''
    SELECT year, genre, appearances, rank
    FROM chart_genre_rank_top
    WHERE year IN (1975, 1995, 2020)
    ORDER BY year, rank
''').df()
display(chk)
print('rank is a direct read of the 03 `rank` column — the bump plots it, does not recompute it.')

### 2b. Share-of-Hot-100 lines (share %, not rank) — the 'how much' framing
Same six genres, but y = **share of that year's chart slots**. Labels at the last
point. This shows magnitude (e.g. the rise of Hip-Hop/Rap's actual share) that a
rank bump flattens.

In [ ]:
share6 = con.execute('SELECT * FROM chart_genre_rank_top ORDER BY year').df()
genres6 = share6.sort_values('year').groupby('genre')['share'].last().sort_values(ascending=False).index.tolist()
wide_share = share6.pivot_table(index='year', columns='genre', values='share', aggfunc='mean').reset_index()
series_share = [{'col': g, 'label': g, 'color': CATEGORICAL_10[i % len(CATEGORICAL_10)]}
                for i, g in enumerate(genres6)]
render_chart({
    'type': 'line',
    'table': wide_share,
    'x_col': 'year',
    'series': series_share,
    'x_axis_label': 'Year',
    'y_axis_label': 'Share of Hot 100 chart slots (%)',
    'y_min': 0,
    'markers': False, 'label_last': True, 'legend': False, 'x_tick_step': 10,
    'title': 'Genre share of the Billboard Hot 100, by year',
    'subtitle': 'Percent of each year\'s chart slots held by the top 6 genres. Coverage thin pre-1970.',
    'source': SRC,
    'filename': 'explore_share_top6_year',
})

**Tradeoff + Kiro recommendation.** The rank bump is the stronger *social* object —
crossings read instantly ('Hip-Hop/Rap passes Rock') and it's the capability we're
building. The share lines are more honest about magnitude and better for the
methodology/README. ➡️ **Kiro lean: lead with the rank bump; keep the share-lines
as the supporting 'how much' chart** (and in the export, where share already lives).

## 3. Year grain vs decade grain

The annual bump (69 years) shows the real crossing *years* but is noisier; the
decade bump (`chart_genre_by_decade`, restricted to the same top-6 and re-ranked)
is a smooth arc that reads cleanly on a phone but loses the exact hand-off year.
Show both.

### 3a. Annual bump (same as 1a, repeated here for the grain comparison)

In [ ]:
bump_from_long(
    top6,
    title='Genre rank on the Billboard Hot 100, by year (annual grain)',
    subtitle='Top 6 genres; annual rank shows the exact crossing years. Coverage thin pre-1970.',
    source=SRC, filename='explore_bump_top6_year_grain', x_tick_step=10,
)

### 3b. Decade bump — smoother arc for social

In [ ]:
dec6 = con.execute('''
    WITH keep AS (
        SELECT * FROM chart_genre_by_decade
        WHERE genre IN (SELECT DISTINCT genre FROM chart_genre_rank_top)
    )
    SELECT decade, genre, appearances, share,
           DENSE_RANK() OVER (PARTITION BY decade ORDER BY appearances DESC) AS rank
    FROM keep
    ORDER BY decade, rank
''').df()
bump_from_long(
    dec6, x_col='decade',
    title='Genre rank on the Billboard Hot 100, by decade',
    subtitle='Top 6 genres by chart appearances, re-ranked within the set per decade.',
    source=SRC, filename='explore_bump_top6_decade', x_tick_step=10,
)

**Tradeoff + Kiro recommendation.** Decade is cleaner but there are only ~7 decade
points, so crossings snap rather than glide, and the 1950s/60s decade points are
built on thin coverage. Annual shows the genuine crossing years (the whole appeal
of a bump) at the cost of some wiggle. ➡️ **Kiro lean: annual grain for the lead
bump** (crossings are the hook), decade as an optional simplified alternate if the
annual reads too busy at phone size.

## 4. Supporting cuts

Two cheap context charts the owner asked for.

### 4a. Metal's trajectory (owner's personal interest)
Metal is **not** in the top-6, so it won't appear in the main bump — but the owner
wants to SEE where it lands. Its own single line of **appearances per year** (peak
**276 in 2001**). Small bucket, clear arc: a late-80s/90s/early-2000s bump, then
decline.

In [ ]:
metal = con.execute('''
    SELECT year, appearances, share, rank
    FROM chart_genre_by_year WHERE genre = 'Metal' ORDER BY year
''').df()
peak = metal.loc[metal['appearances'].idxmax()]
print(f"Metal peak: {int(peak['appearances'])} appearances in {int(peak['year'])} "
      f"(rank #{int(peak['rank'])} that year, {peak['share']}% of the chart)")
render_chart({
    'type': 'line',
    'table': metal,
    'x_col': 'year',
    'series': [{'col': 'appearances', 'label': 'Metal', 'color': c('brown_red')}],
    'x_axis_label': 'Year',
    'y_axis_label': 'Hot 100 chart appearances',
    'y_min': 0, 'markers': True, 'label_last': False, 'legend': False,
    'value_labels': False, 'x_tick_step': 10,
    'title': 'Metal on the Billboard Hot 100, by year',
    'subtitle': 'Chart appearances per year. Metal sits just outside the top 6 (peak 276 in 2001).',
    'source': SRC,
    'filename': 'explore_metal_year',
})

### 4b. Genre share by decade — 100% stacked
A part-of-whole view: each decade's chart slots split across genres. Uses the
shared `stacked_100pct_bars` template on `chart_genre_by_decade` (all 12 genres,
so `Other/Unknown` is visible as its own band — honest about coverage per decade).

In [ ]:
stacked = con.execute('''
    SELECT decade, genre, share
    FROM chart_genre_by_decade ORDER BY decade, genre
''').df()
wide_stacked = stacked.pivot_table(index='decade', columns='genre', values='share',
                                   aggfunc='sum').fillna(0).reset_index()
# group labels must be strings for the stacked template (e.g. '1950s').
wide_stacked['decade'] = wide_stacked['decade'].astype(int).astype(str) + 's'
genre_order = ['Rock', 'R&B/Soul', 'Pop', 'Country', 'Hip-Hop/Rap', 'Dance/Electronic',
               'Latin', 'Folk', 'Jazz', 'Blues', 'Metal', 'Other/Unknown']
genre_order = [g for g in genre_order if g in wide_stacked.columns]
segments = [
    {'col': g, 'label': g,
     'color': (c('gray') if g == 'Other/Unknown' else CATEGORICAL_10[i % len(CATEGORICAL_10)])}
    for i, g in enumerate(genre_order)
]
try:
    render_chart({
        'type': 'stacked_100pct_bars',
        'table': wide_stacked,
        'group_col': 'decade',
        'segments': segments,
        'show_legend': True,
        'title': 'Genre share of the Billboard Hot 100, by decade',
        'subtitle': 'Each decade\'s chart slots split across genres (Other/Unknown in grey). Coverage thin in the 1950s–60s.',
        'source': SRC,
        'filename': 'explore_genre_share_stacked_decade',
    })
except Exception as e:
    # The stacked template signature can vary; if it doesn't take this shape cheaply,
    # fall back to a plain inline table rather than over-building (03 owns chart data).
    print('stacked_100pct_bars did not take this shape cleanly:', e)
    print('Falling back to the numeric share-by-decade matrix (explore via the share-lines above):')
    display(wide_stacked.round(1))

**Note.** The stacked view is secondary — it's a part-of-whole snapshot, not the
rank story. If it renders cleanly it's a nice README/support chart; if not, the
share-lines in §2b already carry the magnitude story, so we don't over-build here.

## Notes for owner — framing decisions to confirm before `06-viz-social`

Four choices settle the lead **bump** (and the capability build in `06`). My
recommendation on each:

1. **`Other/Unknown` handling** — top-6-real only (§1a) vs include-Other (§1b) vs
   top-6-real 1970-start (§1c).
   ➡️ **Recommend §1c (top-6 real, 1970-start)** — honest *and* clean; it simply
   doesn't rank genres in the 1958–69 era where coverage is too thin. If you want
   the full 1958 sweep, use §1a with a visible 'coverage thin pre-1970' caveat.
   Avoid §1b for the social lead (grey not-a-genre line dominates two decades).
2. **Count/rank vs share** — rank bump (§1/§3) vs share-lines (§2b).
   ➡️ **Recommend lead with the rank bump; keep share-lines as support.** The bump
   is the stronger social object (crossings) and the capability we're building;
   share is the honest 'how much' companion for the README/export.
3. **Year vs decade grain** — annual (§3a) vs decade (§3b).
   ➡️ **Recommend annual grain** — the exact crossing years are the whole point of
   a bump. Decade is the fallback if annual reads too busy at phone size.
4. **Final genre line-set** — just the top 6? Add Metal / Latin as context lines?
   ➡️ **Recommend the top 6 for the bump itself** (adding small buckets re-ranks the
   race and clutters it). Show **Metal** as its own small line (§4a — owner interest;
   peaks 276 in 2001, rank #7) and, if wanted, Latin the same way, as separate
   context charts rather than extra bump lines.

**Where Metal landed:** peak **276 appearances in 2001** (rank **#7**, ~5.3% of the
chart that year) — a late-80s→early-2000s hump, just outside the top-6, then
decline. Confirmed inline in §4a.

**Honest limitation (carry into the subtitle/footnote and README):** the bump
ranks are only as good as **genre coverage**, which is thinnest **1958–1969** —
that's exactly when `Other/Unknown` wins (it's #1 in the 1950s, 60s and 90s).
Any 1958-start chart must say so; the 1970-start variant avoids the problem.

⛔ **HOLD `06-viz-social`** (and the new `bump` template build) **until the owner
picks** on the four choices above.

---
## Cleanup
Close the read-only DuckDB connection (single-writer lock).

In [ ]:
con.close()
print('connection closed')